<div dir="rtl" lang="he" style="text-align: right;">

# מפגש 23 — GridSearchCV וקריאת תוצאות
2 שעות הוראה ותרגול ו־1 שעות פרויקט; יחד 135 דקות.

## סיפור המסגרת
דודו משווה שלושה ערכי K. נרשום מראש את המועמדים ונבקש מחיפוש שיטתי לבדוק אותם באותה דרך.

## מטרות
- להגדיר param_grid.
- לקרוא cv_results_.
- להסביר best_params_ ו־refit.

</div>

<div dir="rtl" lang="he" style="text-align: right;">

## המושגים הנחוצים
param_grid מגדיר מועמדים. שם כמו knn__n_neighbors מפנה לפרמטר בתוך שלב Pipeline. GridSearchCV בודק מועמדים באמצעות CV. cv_results_ מכיל תוצאות, ו־best_params_ מחזיר בחירה. refit מתאים את המודל הנבחר לכל Train. mean_test_score הוא אימות פנימי ולא Test חיצוני.

</div>

<div dir="rtl" lang="he" style="text-align: right;">

## דוגמה פתורה לפני קוד
שלושה ערכי K כפול חמישה קיפולים דורשים 15 התאמות מועמדים. כאשר refit=True מתווספת התאמה לכל Train. זה אינו מספר הדוגמאות.

</div>

In [ ]:
candidate_k = [3, 5, 9]
folds = 5
print(len(candidate_k) * folds)

<div dir="rtl" lang="he" style="text-align: right;">

**פלט צפוי:** 15

## מדוע זה חשוב?
מדדים, אימות והכללה משותפים ליחידה 5 ולניסויים ב־DL. שומרים על אותה שיטת הערכה גם כאשר המודל משתנה.

</div>

<div dir="rtl" lang="he" style="text-align: right;">

## יישום מודרך — חלק 1
מקור הקוד: ערכה 37 הקיימת. בהמשך עוברים לתרגול עצמאי באותו נושא.

</div>

<div dir="rtl" lang="he" style="text-align: right;">

### תשתית הנתונים והחלוקה
קראו תחילה את היעד ואת שמות המאפיינים. החלוקה שומרת התאמה בין שורות X ל־y. הפרמטר random_state מאפשר שחזור ואינו משפר דיוק בפני עצמו.

</div>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, classification_report, f1_score
url="https://raw.githubusercontent.com/samiabo75/Data-Science-ML-Course/main/Datasets/penguins.csv"
df=pd.read_csv(url)
data=df.dropna(subset=["species"]).copy()
numeric=["bill_length_mm","bill_depth_mm"]
categorical=["island"]
X=data[numeric+categorical].copy()
y=data["species"].copy()
X_train,X_test,y_train,y_test=train_test_split(
 X,y,test_size=.25,random_state=42,stratify=y)
print("Train:",X_train.shape,"Test:",X_test.shape)

<div dir="rtl" lang="he" style="text-align: right;">

### חישוב ופירוש
קראו את השמות והיחידות, כתבו ציפייה לפני הרצה ובדקו אותה בפלט. הסבירו מה השתנה ומה נשאר קבוע. אין צורך לשנן פקודה בלי להבין את פעולתה.

</div>

In [ ]:
numeric_pipeline=Pipeline([
 ("fill",SimpleImputer(strategy="median")),
 ("scale",StandardScaler())
])
categorical_pipeline=Pipeline([
 ("fill",SimpleImputer(strategy="most_frequent")),
 ("encode",OneHotEncoder(handle_unknown="ignore",sparse_output=False))
])
prepare=ColumnTransformer([
 ("numeric",numeric_pipeline,numeric),
 ("categorical",categorical_pipeline,categorical)
],remainder="drop")
pipe=Pipeline([
 ("prepare",prepare),
 ("knn",KNeighborsClassifier(weights="uniform",metric="euclidean"))
])

<div dir="rtl" lang="he" style="text-align: right;">

### חיפוש מועמדים
קראו את שמות השלבים, רשימת המועמדים ומדד הבחירה. החיפוש מקבל נתוני אימון בלבד; נתוני האימות הפנימיים אינם Test חיצוני.

</div>

In [ ]:
# ההגדרות נבחרו מראש,בלי שימוש בציוני Test
param_grid={"knn__n_neighbors":[3,5,9]}
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
clf=GridSearchCV(
 estimator=pipe,param_grid=param_grid,cv=cv,scoring="f1_macro",
 refit=True,return_train_score=True,n_jobs=1,error_score="raise"
)
clf.fit(X_train,y_train)
print("Best params:",clf.best_params_)
print("Best CV macro-F1:",clf.best_score_)
cv_results=pd.DataFrame(clf.cv_results_)
display(cv_results[[
 "param_knn__n_neighbors","mean_train_score","mean_test_score",
 "std_test_score","rank_test_score"
]].sort_values("rank_test_score"))

<div dir="rtl" lang="he" style="text-align: right;">

### אימות בתוך הפיתוח
ההכנה נמצאת בתוך Pipeline. כל קיפול לומד את פרמטריו מתת־האימון שלו. נקרא את ממוצע האימות לפני שנחליט על מודל.

</div>

In [ ]:
# בודקים קו בסיס באותו מדד CV
baseline=DummyClassifier(strategy="most_frequent")
baseline_cv=cross_val_score(baseline,X_train,y_train,cv=cv,scoring="f1_macro")
baseline.fit(X_train,y_train)
print("Baseline CV macro-F1:",baseline_cv.mean())

# refit=True כבר התאים את המודל הנבחר לכל Train
predictions=clf.predict(X_test)
baseline_predictions=baseline.predict(X_test)
print("Final model Test macro-F1:",f1_score(y_test,predictions,average="macro"))
print("Baseline Test macro-F1:",f1_score(y_test,baseline_predictions,average="macro"))
print("Final model Test Accuracy:",accuracy_score(y_test,predictions))
print(classification_report(y_test,predictions,zero_division=0))

<div dir="rtl" lang="he" style="text-align: right;">

## עצירה לפני המעבדה
מה שונה בין best_params_ לבין ממוצע Scaler?

**התוצר הנדרש:** במעבדת 37 מבצעים מועמדים, חיפוש וקריאת פלט. ההכנה המשולבת היא חזרה ממפגש 14.

</div>